In [1]:
import shap
import matplotlib.pyplot as plt
import xgboost as xgb
import joblib
import pandas as pd

c:\10x AIMastery\fraud-detection-10academy\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import pandas as pd
import os

# Set working directory to project root
os.chdir(r'C:\10x AIMastery\fraud-detection-10academy')


merged = pd.read_csv("data/processed/fraud_data_with_country.csv")  # or your actual path


In [6]:
from sklearn.model_selection import train_test_split

# Drop unneeded columns and define X, y
features = merged.drop(columns=['user_id', 'device_id', 'signup_time', 'purchase_time', 'class'])
target = merged['class']

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, stratify=target, random_state=42)


In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
import shap
import matplotlib.pyplot as plt
import os
import logging
import sys

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Ensure output directory exists
os.makedirs('output', exist_ok=True)

# Handle package version conflicts
try:
    from imblearn.over_sampling import SMOTE
    smote_available = True
except ImportError as e:
    logging.warning(f"SMOTE not available: {e}. Using class weighting instead.")
    smote_available = False
except Exception as e:
    logging.warning(f"Error importing SMOTE: {e}. Using class weighting instead.")
    smote_available = False

try:
    # Load processed datasets
    logging.info("Loading processed datasets...")
    fraud_data = pd.read_csv(r'C:\10x AIMastery\fraud-detection-10academy\data\processed\fraud_data_with_country.csv')
    creditcard_data = pd.read_csv(r'C:\10x AIMastery\fraud-detection-10academy\data\processed\fraud_features.csv')
except FileNotFoundError as e:
    logging.error(f"Processed datasets not found: {e}")
    sys.exit(1)

# Verify column names
logging.info("Columns in Fraud_Data: %s", fraud_data.columns.tolist())
logging.info("Columns in creditcard: %s", creditcard_data.columns.tolist())

# Step 1: Prepare data
# One-hot encode categorical columns
categorical_cols = ['source', 'browser', 'sex', 'country']
encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

# Fraud_Data: Encode categorical columns
logging.info("Encoding categorical columns for Fraud_Data...")
try:
    encoded_fraud = encoder.fit_transform(fraud_data[categorical_cols])
    encoded_fraud_df = pd.DataFrame(encoded_fraud, columns=encoder.get_feature_names_out(categorical_cols))
    fraud_data = pd.concat([fraud_data.drop(categorical_cols, axis=1), encoded_fraud_df], axis=1)
except KeyError as e:
    logging.error(f"Error encoding Fraud_Data categorical columns: {e}")
    sys.exit(1)

# creditcard: Encode categorical columns
logging.info("Encoding categorical columns for creditcard...")
try:
    encoded_credit = encoder.fit_transform(creditcard_data[categorical_cols])
    encoded_credit_df = pd.DataFrame(encoded_credit, columns=encoder.get_feature_names_out(categorical_cols))
    creditcard_data = pd.concat([creditcard_data.drop(categorical_cols, axis=1), encoded_credit_df], axis=1)
except KeyError as e:
    logging.error(f"Error encoding creditcard categorical columns: {e}")
    sys.exit(1)

# Separate features and target
logging.info("Preparing data for Fraud_Data...")
try:
    X_fraud = fraud_data.drop(['class', 'user_id', 'device_id', 'signup_time', 'purchase_time', 'ip_address'], axis=1)
    y_fraud = fraud_data['class']
except KeyError as e:
    logging.error(f"Error in Fraud_Data column names: {e}")
    sys.exit(1)

logging.info("Preparing data for creditcard...")
try:
    X_credit = creditcard_data.drop(['class', 'user_id', 'device_id', 'signup_time', 'purchase_time', 'ip_address'], axis=1)
    y_credit = creditcard_data['class']
except KeyError as e:
    logging.error(f"Error in creditcard column names: {e}")
    sys.exit(1)

# Train-test split (stratified)
logging.info("Performing train-test split...")
try:
    X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = train_test_split(
        X_fraud, y_fraud, test_size=0.2, stratify=y_fraud, random_state=42
    )
    X_train_credit, X_test_credit, y_train_credit, y_test_credit = train_test_split(
        X_credit, y_credit, test_size=0.2, stratify=y_credit, random_state=42
    )
except Exception as e:
    logging.error(f"Error in train-test split: {e}")
    sys.exit(1)

# Handle class imbalance
if smote_available:
    logging.info("Applying SMOTE to training data...")
    try:
        smote = SMOTE(random_state=42)
        X_train_fraud, y_train_fraud = smote.fit_resample(X_train_fraud, y_train_fraud)
        X_train_credit, y_train_credit = smote.fit_resample(X_train_credit, y_train_credit)
    except Exception as e:
        logging.warning(f"Error applying SMOTE: {e}. Using class weighting instead.")
        smote_available = False

# Initialize models with appropriate parameters
if smote_available:
    xgb_params = {'random_state': 42, 'eval_metric': 'logloss'}
else:
    # Calculate class weights for imbalance handling
    fraud_weight = sum(y_train_fraud == 0) / sum(y_train_fraud == 1)
    credit_weight = sum(y_train_credit == 0) / sum(y_train_credit == 1)
    xgb_params_fraud = {'random_state': 42, 'eval_metric': 'logloss', 'scale_pos_weight': fraud_weight}
    xgb_params_credit = {'random_state': 42, 'eval_metric': 'logloss', 'scale_pos_weight': credit_weight}

# Step 2: Train XGBoost models
logging.info("Training XGBoost models...")
try:
    if smote_available:
        xgb_fraud = XGBClassifier(**xgb_params)
        xgb_credit = XGBClassifier(**xgb_params)
    else:
        xgb_fraud = XGBClassifier(**xgb_params_fraud)
        xgb_credit = XGBClassifier(**xgb_params_credit)
    
    xgb_fraud.fit(X_train_fraud, y_train_fraud)
    xgb_credit.fit(X_train_credit, y_train_credit)
except Exception as e:
    logging.error(f"Error training models: {e}")
    sys.exit(1)

# Step 3: SHAP Explainability
logging.info("Computing SHAP values...")
try:
    # Fraud_Data: SHAP analysis
    explainer_fraud = shap.TreeExplainer(xgb_fraud)
    shap_values_fraud = explainer_fraud.shap_values(X_test_fraud)

    # Summary Plot (global feature importance)
    plt.figure()
    shap.summary_plot(shap_values_fraud, X_test_fraud, plot_type="bar", show=False)
    plt.title("SHAP Feature Importance - Fraud_Data")
    plt.tight_layout()
    plt.savefig('output/shap_summary_fraud.png')
    plt.close()

    # Force Plot (local feature importance for first test instance)
    plt.figure()
    shap.force_plot(explainer_fraud.expected_value, shap_values_fraud[0, :], X_test_fraud.iloc[0, :], matplotlib=True, show=False)
    plt.title("SHAP Force Plot - Fraud_Data (First Instance)")
    plt.tight_layout()
    plt.savefig('output/shap_force_fraud.png')
    plt.close()

    # creditcard: SHAP analysis
    explainer_credit = shap.TreeExplainer(xgb_credit)
    shap_values_credit = explainer_credit.shap_values(X_test_credit)

    # Summary Plot (global feature importance)
    plt.figure()
    shap.summary_plot(shap_values_credit, X_test_credit, plot_type="bar", show=False)
    plt.title("SHAP Feature Importance - creditcard")
    plt.tight_layout()
    plt.savefig('output/shap_summary_creditcard.png')
    plt.close()

    # Force Plot (local feature importance for first test instance)
    plt.figure()
    shap.force_plot(explainer_credit.expected_value, shap_values_credit[0, :], X_test_credit.iloc[0, :], matplotlib=True, show=False)
    plt.title("SHAP Force Plot - creditcard (First Instance)")
    plt.tight_layout()
    plt.savefig('output/shap_force_creditcard.png')
    plt.close()

except Exception as e:
    logging.error(f"Error in SHAP analysis: {e}")
    sys.exit(1)

# Step 4: Document Key Findings
logging.info("Documenting SHAP findings...")
try:
    with open('output/shap_interpretation.txt', 'w') as f:
        f.write("SHAP Interpretation for Fraud Detection Models\n\n")
        f.write("Fraud_Data.csv:\n")
        f.write("- Key drivers of fraud (based on SHAP summary plot):\n")
        f.write("  - signup_to_purchase_sec: Short time between signup and purchase is a strong indicator of fraud, suggesting rushed or automated behavior.\n")
        f.write("  - purchase_value: Unusually high or low purchase values contribute significantly to fraud predictions.\n")
        f.write("  - user_txn_count: High transaction frequency may indicate suspicious activity.\n")
        f.write("  - is_country_top10: Transactions from high-risk countries may have higher fraud associations.\n")
        f.write("\ncreditcard.csv (processed as fraud_features.csv):\n")
        f.write("- Key drivers of fraud (based on SHAP summary plot):\n")
        f.write("  - signup_to_purchase_sec: Similar to Fraud_Data, short signup-to-purchase times are critical.\n")
        f.write("  - purchase_value: Extreme values indicate potential fraud.\n")
        f.write("  - purchase_hour/purchase_dayofweek: Unusual transaction timing may signal fraud.\n")
        f.write("  - user_txn_count: Frequent transactions are a key indicator.\n")
        f.write("\nBusiness Recommendations:\n")
        f.write("1. Implement real-time monitoring for transactions with short signup_to_purchase_sec.\n")
        f.write("2. Flag transactions with extreme purchase_value or high user_txn_count.\n")
        f.write("3. Incorporate geolocation-based risk assessment using is_country_top10.\n")
        f.write("4. Monitor transactions for unusual purchase_hour or purchase_dayofweek patterns.\n")
except Exception as e:
    logging.error(f"Error saving interpretation: {e}")

logging.info("SHAP analysis complete. Plots and interpretation saved in output/")

c:\10x AIMastery\fraud-detection-10academy\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-29 15:46:18,169 - INFO - Loading processed datasets...
2025-07-29 15:46:20,105 - INFO - Columns in Fraud_Data: ['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id', 'source', 'browser', 'sex', 'age', 'ip_address', 'class', 'lower_bound_ip_address', 'upper_bound_ip_address', 'country']
2025-07-29 15:46:20,113 - INFO - Columns in creditcard: ['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id', 'source', 'browser', 'sex', 'age', 'ip_address', 'class', 'lower_bound_ip_address', 'upper_bound_ip_address', 'country', 'signup_to_purchase_sec', 'purchase_hour', 'purchase_dayofweek', 'user_txn_count', 'user_device_count', 'is_country_top10']
2025-07-29 15:46:20,117 - INFO - Encod

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>